# Trees and Model Stability

Trees are notorious for being **unstable**: Small changes in the data can lead to noticeable or large changes in the tree. We're going to explore this phenomenon.

1. Check out the data set `heart_failure` and appropriately clean it.
2. Perform a train-test split for a specific seed (save the seed for reproducibility). Fit a tree and a linear model on the training data and evaluate their performance on the test data. Set aside the predictions these models make.
3. Repeat step 2 for three to five different seeds (save the seeds for reproducibility). How different are the trees that you get? Your linear model coefficients?. Set aside the predictions these models make.
4. Typically, you would see the trees changing what appears to be a non-trivial amount, while the linear model coefficients don't vary nearly as much. Often, the changes appear substantial.
But are they? Instead of focusing on the tree or model coefficients, do three things:
    
    a. Make scatterplots of the predicted values on the test set from question 2 against the predicted values for the alternative models from part 3, separately for your trees and linear models. Do they appear reasonably similar?
    
    b. Compute the correlation between your model in part 2 and your alternative models in part 3, separately for your trees and linear models. Are they highly correlated or not?
    
    c. Run a simple linear regression of the predicted values on the test set from the alternative models on the predicted values from question 2, separately for your trees and linear models. Is the intercept close to zero? Is the slope close to 1? Is the $R^2$ close to 1?

5. Do linear models appear to have similar coefficients and predictions across train/test splits? Do trees?
6. True or false for this dataset, and explain: "Even if the models end up having a substantially different appearance, the predictions they generate are often very similar."

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** [To be provided]

**Date:** [To be provided]

In [ ]:
# Your Phase 1 solution to the problem goes here.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

URL = ("https://raw.githubusercontent.com/elijahschoeniger-cpu/L03-tree-stability/"
       "main/data/heart_failure/heart_failure_clinical_records_dataset.csv")

df = pd.read_csv(URL)

#1
print(df.shape)                      # (299, 13)
print(df.dtypes)
print("Missing values:", df.isna().sum().sum())   # 0
print("Duplicate rows:", df.duplicated().sum())   # 0
print(df.describe().T)

df = df.rename(columns={"DEATH_EVENT": "death_event"})

binary_cols = ["anaemia", "diabetes", "high_blood_pressure", "sex", "smoking", "death_event"]
for c in binary_cols:
    assert set(df[c].unique()) <= {0, 1}, f"{c} has non-binary values"
    df[c] = df[c].astype(int)

# Two rows have age = 60.667
df["age"] = df["age"].round().astype(int)

# No NaNs, but the source filled gaps with summary stats:
#   platelets = 263358.03 (the column mean) in 25 rows
#   creatinine_phosphokinase = 582 in 47 rows (a suspicious spike at the median)
# Keep the values, but add flags so the models can tell them apart.
df["platelets_imputed"] = np.isclose(df["platelets"], 263358.03).astype(int)
df["cpk_imputed"] = (df["creatinine_phosphokinase"] == 582).astype(int)

# patients who died were followed for less time median 44.5 vs 172 days, encodes the outcome rather than predicting it.
df = df.drop(columns="time")

# 6. Log-transform heavy because of the heavy right skew helps the linear model
# Skew: CPK ~4.5, serum_creatinine ~4.5. Trees don't care; logistic regression does.
df["log_cpk"] = np.log(df["creatinine_phosphokinase"])
df["log_serum_creatinine"] = np.log(df["serum_creatinine"])
df = df.drop(columns=["creatinine_phosphokinase", "serum_creatinine"])

df["platelets"] = df["platelets"] / 1000

X = df.drop(columns="death_event")
y = df["death_event"]

print(df.head())
print("Class balance:\n", y.value_counts(normalize=True))   # ~68% survived / 32% died


#Q2
print("Q2")
from sklearn.model_selection import train_test_split
x = X
y = df['death_event']
train_test_split(x, y, test_size=0.2, random_state=12)
train_test_split(x, y, test_size=0.2, random_state=13)
train_test_split(x, y, test_size=0.2, random_state=15)
train_test_split(x, y, test_size=0.2, random_state=16)
train_test_split(x, y, test_size=0.2, random_state=17)

(299, 13)
age                         float64
anaemia                       int64
creatinine_phosphokinase      int64
diabetes                      int64
ejection_fraction             int64
high_blood_pressure           int64
platelets                   float64
serum_creatinine            float64
serum_sodium                  int64
sex                           int64
smoking                       int64
time                          int64
DEATH_EVENT                   int64
dtype: object
Missing values: 0
Duplicate rows: 0
                          count           mean           std      min  \
age                       299.0      60.833893     11.894809     40.0   
anaemia                   299.0       0.431438      0.496107      0.0   
creatinine_phosphokinase  299.0     581.839465    970.287881     23.0   
diabetes                  299.0       0.418060      0.494067      0.0   
ejection_fraction         299.0      38.083612     11.834841     14.0   
high_blood_pressure       299.0   

[     age  anaemia  diabetes  ejection_fraction  high_blood_pressure  \
 207   85        0         0                 38                    0   
 233   53        1         0                 38                    0   
 116   60        1         1                 60                    1   
 232   40        1         0                 35                    0   
 48    80        1         0                 20                    1   
 ..   ...      ...       ...                ...                  ...   
 278   50        1         1                 30                    0   
 240   70        0         1                 35                    1   
 143   61        1         0                 40                    1   
 241   65        0         1                 30                    0   
 111   55        0         0                 35                    0   
 
      platelets  serum_sodium  sex  smoking  platelets_imputed  cpk_imputed  \
 207      186.0           136    1        0            

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]